# 22. AI2D — **GraphColBERT**: FiLM-фьюжн структуры диаграммы и ColBERT late-interaction

Адаптация архитектуры GraphColBERT под **MCQ** (AI2D: вопрос + диаграмма + 4 варианта,
выбрать правильный). Метрика — **accuracy** (argmax по вариантам).

### Идея
- **ColBERT** (Khattab & Zaharia, 2020) кодирует текст **по-токенно**; релевантность —
  `MaxSim(q, d) = Σ_i max_j (q_i · d_j)`. Сильный лексический матчинг вопроса и варианта,
  но ColBERT **слеп к структуре диаграммы** — не знает, что на схеме нарисовано и где.
- **GNN** (`GraphEncoderV2`, чекпоинт `ai2d_hybrid_manual_full`) строит граф диаграммы
  (узлы = OpenCV-фигуры + OCR-подписи, типизированные kNN-рёбра по геометрии) и пулингом
  даёт **структурный вектор диаграммы** `z_img ∈ ℝ²⁵⁶`. Но схлопывает всё в один эмбеддинг,
  теряя по-токенную точность сопоставления текста.

### Мост: FiLM-модуляция токенов вариантов структурой диаграммы
В отличие от DocVQA (там узел графа ↔ OCR-строка-кандидат 1:1), в MCQ кандидаты — это
**4 варианта ответа**, а граф описывает **диаграмму целиком**. Поэтому структурным
conditioning-вектором служит **пулинг графа** `z_img`, которым модулируются ColBERT-токены
каждого варианта (канонический FiLM с глобальным conditioning):

```
z_img        = pool(GNN(graph_диаграммы))     # ℝ²⁵⁶, заморожен
γ, β         = MLP(z_img)                      # ℝ⁹⁶ каждый  (FiLM-параметры)
Q            = ColBERT(вопрос + OCR-текст)     # [m, 96]   query с grounding
D_i          = ColBERT(вариант_i)              # [t_i, 96]
D_i'         = D_i · (1 + γ) + β               # модуляция токенов варианта
score_i      = Σ_q max_d (q · D_i')            # MaxSim по обогащённым токенам
pred         = argmax_i score_i
```

Матчинг остаётся по-токенным (ColBERT), но геометрия сопоставления теперь обусловлена
структурой диаграммы (GNN). **Оба бэкбона заморожены**, учится только мост (две MLP + темп.).
При инициализации `γ=β=0` ⇒ `D_i' = D_i`, т.е. модель стартует **ровно как plain ColBERT**
и обучением может только улучшаться.

### Grounding query
Query = `вопрос + OCR-текст диаграммы` (реальные подписи на схеме: FACE, NOSE, ...), что
напрямую помогает разрешить правильный вариант. OCR берётся из `ocr_v2` (conf ≥ 35).

### Обучение и метрика
AI2D — **MCQ**, метрика **accuracy**. Позитив = `correct_option_idx` (есть у каждого вопроса).
Лосс — кросс-энтропия по 4 вариантам (InfoNCE внутри вопроса). Ранняя остановка по val accuracy.

**Сравниваем три модели на одних входах:** graph-only · plain ColBERT · **GraphColBERT (FiLM)**.

In [1]:
import sys
from pathlib import Path

p = Path.cwd()
while p != p.parent:
    if (p / 'src' / 'vqa_retrieval').exists() and (p / 'notebooks').exists():
        break
    p = p.parent
ROOT = p
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

DATA_ROOT  = ROOT.parent                       # пути в манифесте — относительно неё
AI2D_DIR   = DATA_ROOT / 'ai2d'
MANIFEST   = AI2D_DIR / 'prepared_v2' / 'manifest_hybrid.jsonl'
CKPT_GRAPH = ROOT / 'runs' / 'ai2d_hybrid_manual_full' / 'checkpoint_best.pt'
OUTPUT_DIR = ROOT / 'runs' / 'ai2d_graphcolbert_film'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:       ', ROOT)
print('MANIFEST:   ', MANIFEST, '->', MANIFEST.exists())
print('graph ckpt: ', CKPT_GRAPH.exists())

ROOT:        C:\Users\Jet\Desktop\data\ai2d_vqa_clean
MANIFEST:    C:\Users\Jet\Desktop\data\ai2d\prepared_v2\manifest_hybrid.jsonl -> True
graph ckpt:  True


In [2]:
# scikit-learn должен быть 1.6.1, sentence-transformers 5.3.0 — иначе import падает
# (stack overflow) или конфликтует с pylate. См. память проекта.
import json, random, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm

# КРИТИЧНО (Windows): нативные либы pyarrow (его тянет sentence_transformers -> datasets)
# дают access violation / КРАШ ЯДРА, если грузятся ПОСЛЕ инициализации CUDA-контекста.
# Поэтому весь тяжёлый нативный стек импортируем ЗДЕСЬ, ДО первого обращения к CUDA
# (torch.cuda.is_available() ниже) и до тяжёлых ячеек 5–6. См. память проекта.
try:
    import pyarrow
    import datasets
except Exception:
    pass
import cv2
import timm
import sentence_transformers  # noqa: F401  (тянет datasets -> pyarrow)
import torch_geometric

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# CUDA инициализируется ТОЛЬКО здесь — строго после тяжёлых импортов выше.
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
COLBERT_NAME = 'answerdotai/answerai-colbert-small-v1'

# Данные (None = весь split). Первый прогон — на подвыборке; потом ставим None.
MAX_TRAIN    = None
MAX_VAL      = None
MAX_TEST     = None      # весь test (3088) для итогового числа
OCR_MIN_CONF = 35.0
MAX_OCR_WORDS = 64       # потолок длины OCR-grounding в query
KNN_K        = 4

# Обучение моста
EPOCHS       = 200
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 30
FILM_HIDDEN  = 128
DROPOUT      = 0.1

print('Device:', DEVICE)

C:\Users\Jet\Desktop\data\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [3]:
def load_manifest(path):
    by = {'train': [], 'val': [], 'test': []}
    with open(path, encoding='utf-8') as f:
        for line in f:
            d = json.loads(line)
            sp = d.get('split')
            if sp not in by:
                continue
            opts = [str(o).strip() for o in d.get('options', []) if str(o).strip()]
            if len(opts) < 2:
                continue
            ci = int(d.get('correct_option_idx', 0))
            ci = max(0, min(ci, len(opts) - 1))
            by[sp].append({
                'question':   d['question'],
                'options':    opts,
                'correct':    ci,
                'image_path': str(DATA_ROOT / d['image_path']),
                'ocr_path':   str(DATA_ROOT / d['ocr_v2_path']),
            })
    return by

_by = load_manifest(MANIFEST)
def _cap(rows, n):
    return rows if n is None else rows[:n]
train_samples = _cap(_by['train'], MAX_TRAIN)
val_samples   = _cap(_by['val'],   MAX_VAL)
test_samples  = _cap(_by['test'],  MAX_TEST)
print(f'train: {len(train_samples)} | val: {len(val_samples)} | test: {len(test_samples)}')
print('пример:', val_samples[0]['question'], '->', val_samples[0]['options'],
      '| correct =', val_samples[0]['correct'])

train: 11145 | val: 1268 | test: 3088
пример: Which image represents Fagonia indica? -> ['D', 'E', 'A', 'B'] | correct = 2


In [4]:
def load_ocr_text(ocr_path, min_conf=OCR_MIN_CONF, max_words=MAX_OCR_WORDS):
    """Склеиваем OCR-строки диаграммы (conf >= порога) в один grounding-текст."""
    try:
        d = json.loads(Path(ocr_path).read_text(encoding='utf-8'))
    except Exception:
        return ''
    items = d.get('lines') or d.get('words') or []
    toks = []
    for it in items:
        t = str(it.get('text', '')).strip()
        if t and float(it.get('conf', 0.0)) >= min_conf:
            toks.append(t)
    return ' '.join(toks[:max_words])

def build_query_text(item):
    ocr = load_ocr_text(item['ocr_path'])
    return f"{item['question']} {ocr}".strip()

print('пример query:', build_query_text(val_samples[0])[:160])

пример query: Which image represents Fagonia indica? SCAG OA One Se ‘ SS. WS aid Ges fy att 04 w ae & A Ye WS sf, in BN 4 Citrullus colocynthis Limonium axililare Ls Qa “A “S


---
## Структурная ветка: пулинг GNN-графа диаграммы

`GraphEncoderV2` (перенос с AI2D-чекпоинта `ai2d_hybrid_manual_full`) строит граф диаграммы
(OpenCV-фигуры + OCR-подписи, типизированные kNN-рёбра) и пулингом даёт `z_img ∈ ℝ²⁵⁶`.
Этот же вектор служит conditioning-входом FiLM. Для baseline graph-only вариант скорится как
в исходном гибриде: `cos(z_img, text_proj(вариант))`.

In [5]:
import cv2
from vqa_retrieval.graph_builder_v2 import NodeFeaturizerV2, GraphEncoderV2, build_graph_v2

graph_ckpt = torch.load(CKPT_GRAPH, map_location=DEVICE, weights_only=False)
cfg = graph_ckpt['model_config']

featurizer = NodeFeaturizerV2(
    device=DEVICE,
    vision_model_name=cfg['vision_model_name'],
    text_model_name=cfg['text_model_name'],
)
gnn = GraphEncoderV2(
    in_dim=cfg['in_dim'], hidden_dim=cfg['hidden_dim'], out_dim=cfg['out_dim'],
    use_attn_pool=cfg['use_attn_pool'],
).to(DEVICE)

gnn.load_state_dict(graph_ckpt['gnn_state_dict']); gnn.eval()
for _p in gnn.parameters():
    _p.requires_grad_(False)

text_proj = nn.Sequential(
    nn.Linear(featurizer.text_dim, cfg['hidden_dim']), nn.GELU(),
    nn.Linear(cfg['hidden_dim'], cfg['out_dim']),
).to(DEVICE)
text_proj.load_state_dict(graph_ckpt['text_proj_state_dict']); text_proj.eval()

G_DIM = cfg['out_dim']
_KNN  = cfg.get('extract_knn_k', KNN_K)
print('Graph model loaded | G_DIM =', G_DIM, '| knn_k =', _KNN)

@torch.no_grad()
def diagram_embedding(item):
    """Пулинг графа диаграммы -> нормированный z_img [G_DIM]."""
    graph = build_graph_v2(item['image_path'], featurizer,
                           ocr_path=item['ocr_path'], knn_k=_KNN).to(DEVICE)
    return F.normalize(gnn(graph), dim=-1).reshape(-1)

@torch.no_grad()
def graph_option_scores(options, z_img):
    """graph-only baseline: cos(z_img, text_proj(вариант))."""
    raw = featurizer.text_enc.encode(options, convert_to_tensor=True,
                                     normalize_embeddings=False).to(DEVICE)
    z_opt = F.normalize(text_proj(raw), dim=-1)
    return (z_opt @ z_img).float().cpu().numpy()

Graph model loaded | G_DIM = 256 | knn_k = 4


C:\Users\Jet\Desktop\data\ai2d_vqa_clean\src\vqa_retrieval\graph_builder_v2.py:330: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn=gate_nn)


---
## Ветка late-interaction: ColBERT по-токенно

pylate ставим с `--no-deps` (иначе `fast-plaid` ломает CUDA-torch — см. память проекта).
Query = `вопрос + OCR`; документы = 4 варианта. `encode` возвращает список тензоров
`[n_токенов, 96]`.

In [6]:

from pylate import models

colbert = models.ColBERT(model_name_or_path=COLBERT_NAME, device=DEVICE)
D_CB = int(colbert.encode(['probe'], is_query=False, convert_to_tensor=True)[0].shape[-1])
print('ColBERT ready | D_CB =', D_CB)

@torch.no_grad()
def encode_query(text):
    return colbert.encode([text], is_query=True, convert_to_tensor=True)[0]

@torch.no_grad()
def encode_options(options):
    embs = colbert.encode(options, is_query=False, convert_to_tensor=True)
    return [e for e in embs]

def maxsim(Q, D_i):
    return (Q @ D_i.T).max(dim=1).values.sum()

@torch.no_grad()
def plain_colbert_scores(Q, D_list):
    return torch.stack([maxsim(Q, d) for d in D_list])

No sentence-transformers model found with name answerdotai/answerai-colbert-small-v1.


ColBERT ready | D_CB = 96


---
## Мост: FiLM-модуляция токенов вариантов структурой диаграммы

`z_img` (пулинг графа) задаёт `γ, β`, которыми модулируются ColBERT-токены **каждого**
варианта; затем обычный MaxSim. Последний слой FiLM инициализируется нулями ⇒ `γ=β=0` ⇒
`D_i' = D_i` ⇒ на старте поведение **идентично plain ColBERT**.

In [7]:
class GraphFiLMColBERT(nn.Module):
    def __init__(self, g_dim, d_cb, hidden=128, dropout=0.1):
        super().__init__()
        self.d_cb = d_cb
        self.film = nn.Sequential(
            nn.Linear(g_dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, 2 * d_cb),
        )
        nn.init.zeros_(self.film[-1].weight)   # старт: γ=β=0 -> D_i' = D_i (= plain ColBERT)
        nn.init.zeros_(self.film[-1].bias)
        self.temp = nn.Parameter(torch.tensor(1.0))

    def modulate(self, D_i, z_img):
        gamma, beta = self.film(z_img).chunk(2, dim=-1)
        return D_i * (1.0 + gamma) + beta

    def option_score(self, Q, D_i, z_img):
        Dp  = self.modulate(D_i, z_img)
        sim = Q @ Dp.T
        return sim.max(dim=1).values.sum() * self.temp.clamp(min=0.05)

    def forward(self, Q, D_list, z_img):
        """Q [m,d_cb]; D_list список [t_i,d_cb]; z_img [g_dim] -> скоры вариантов [n_opt]."""
        return torch.stack([self.option_score(Q, D_list[i], z_img) for i in range(len(D_list))])

bridge = GraphFiLMColBERT(G_DIM, D_CB, hidden=FILM_HIDDEN, dropout=DROPOUT).to(DEVICE)
n_par = sum(p.numel() for p in bridge.parameters() if p.requires_grad)
print(f'FiLM bridge: {n_par:,} обучаемых параметров (бэкбоны заморожены)')

FiLM bridge: 57,665 обучаемых параметров (бэкбоны заморожены)


---
## Предрасчёт примеров

Каждый вопрос **один раз** прогоняем через замороженные GNN и ColBERT и кешируем на CPU
(`fp16`): структурный `z_img`, токены вопроса `Q`, токены 4 вариантов `D_list`, тексты
вариантов, индекс правильного и graph-only скоры (для baseline).

In [8]:
def precompute(samples, split):
    cache = OUTPUT_DIR / f'precomp_{split}.pkl'
    if cache.exists():
        print('загружаю кэш', cache.name)
        return pickle.loads(cache.read_bytes())
    rows = []
    for s in tqdm(samples, desc=f'precompute {split}'):
        try:
            z_full = diagram_embedding(s)                          # [G_DIM] на DEVICE
        except Exception as e:
            rows.append(None); continue
        gscore = graph_option_scores(s['options'], z_full)
        Q  = encode_query(build_query_text(s)).half().cpu()
        Dl = [d.half().cpu() for d in encode_options(s['options'])]
        rows.append({
            'z_img':   z_full.half().cpu(),
            'Q':       Q,
            'D':       Dl,
            'options': s['options'],
            'correct': s['correct'],
            'graph':   gscore.astype('float32'),
        })
    cache.write_bytes(pickle.dumps(rows))
    return rows

train_rows = precompute(train_samples, 'train')
val_rows   = precompute(val_samples,   'val')
test_rows  = precompute(test_samples,  'test')

train_ok = [r for r in train_rows if r is not None]
val_ok   = [r for r in val_rows   if r is not None]
test_ok  = [r for r in test_rows  if r is not None]
print(f'train: {len(train_ok)}/{len(train_rows)} | val: {len(val_ok)}/{len(val_rows)} '
      f'| test: {len(test_ok)}/{len(test_rows)}')

загружаю кэш precomp_train.pkl
загружаю кэш precomp_val.pkl
загружаю кэш precomp_test.pkl
train: 4000/4000 | val: 800/800 | test: 3088/3088


---
## Обучение моста

Кросс-энтропия по 4 вариантам (InfoNCE внутри вопроса): позитив = `correct_option_idx`.
Backbone'ы заморожены — учится только FiLM-мост. Ранняя остановка по **val accuracy**.

In [9]:
def to_dev(r):
    Q = r['Q'].float().to(DEVICE)
    D = [d.float().to(DEVICE) for d in r['D']]
    z = r['z_img'].float().to(DEVICE)
    return Q, D, z

def film_scores(r):
    Q, D, z = to_dev(r)
    with torch.no_grad():
        return bridge(Q, D, z).cpu().numpy()

def plain_scores(r):
    Q, D, _ = to_dev(r)
    return plain_colbert_scores(Q, D).cpu().numpy()

def graph_scores(r):
    return r['graph']

@torch.no_grad()
def accuracy(rows, score_fn):
    correct, n = 0, 0
    for r in rows:
        if r is None:
            continue
        n += 1
        if int(np.argmax(score_fn(r))) == r['correct']:
            correct += 1
    return correct / max(n, 1)

opt = torch.optim.AdamW(bridge.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
best_acc, best_state, bad = -1.0, None, 0

for epoch in range(1, EPOCHS + 1):
    bridge.train()
    random.shuffle(train_ok)
    run = 0.0
    for r in tqdm(train_ok, desc=f'epoch {epoch}', leave=False):
        Q, D, z = to_dev(r)
        scores = bridge(Q, D, z).unsqueeze(0)
        loss = F.cross_entropy(scores, torch.tensor([r['correct']], device=DEVICE))
        opt.zero_grad(); loss.backward(); opt.step()
        run += loss.item()
    bridge.eval()
    va = accuracy(val_ok, film_scores)
    print(f'epoch {epoch:2d} | loss={run/max(len(train_ok),1):.4f} | val acc={va:.4f}', end='')
    if va > best_acc:
        best_acc, bad = va, 0
        best_state = {k: v.detach().cpu().clone() for k, v in bridge.state_dict().items()}
        torch.save({'bridge_state_dict': best_state, 'val_acc': va, 'epoch': epoch,
                    'G_DIM': G_DIM, 'D_CB': D_CB, 'film_hidden': FILM_HIDDEN},
                   OUTPUT_DIR / 'graphcolbert_film_best.pt')
        print('  <- best (сохранено)')
    else:
        bad += 1; print(f'  (no-improve {bad}/{PATIENCE})')
        if bad >= PATIENCE:
            print('ранняя остановка'); break

if best_state is not None:
    bridge.load_state_dict(best_state)
print(f'best val acc = {best_acc:.4f}')

epoch  1 | loss=1.4132 | val acc=0.3275  <- best (сохранено)


epoch  2 | loss=1.3793 | val acc=0.2938  (no-improve 1/30)


epoch  3 | loss=1.3445 | val acc=0.3250  (no-improve 2/30)


epoch  4 | loss=1.3241 | val acc=0.3300  <- best (сохранено)


epoch  5 | loss=1.3073 | val acc=0.3250  (no-improve 1/30)


...

epoch 44 | loss=0.8800 | val acc=0.3187  (no-improve 26/30)


epoch 45 | loss=0.8786 | val acc=0.3162  (no-improve 27/30)


epoch 46 | loss=0.8675 | val acc=0.3275  (no-improve 28/30)


epoch 47 | loss=0.8686 | val acc=0.3125  (no-improve 29/30)


epoch 48 | loss=0.8525 | val acc=0.3187  (no-improve 30/30)
ранняя остановка
best val acc = 0.3525


---
## Оценка и сравнение

Три модели на **test**:
- **graph-only** — `cos(z_img, text_proj(вариант))` (структура диаграммы, перенос с AI2D),
- **plain ColBERT** — MaxSim(вопрос+OCR, вариант) без модуляции,
- **GraphColBERT (FiLM)** — обученный мост.

Честный исход: FiLM должен быть ≥ plain ColBERT (он его обобщает). Random baseline = 0.25
(4 варианта).

In [ ]:
import pandas as pd

bridge.eval()
acc_graph = accuracy(test_ok, graph_scores)
acc_plain = accuracy(test_ok, plain_scores)
acc_film  = accuracy(test_ok, film_scores)

rows_tbl = [
    ('Random baseline (1/4)',               0.25),
    ('graph-only (GNN, transfer AI2D)',     acc_graph),
    ('plain ColBERT (вопрос+OCR ↔ вариант)', acc_plain),
    ('GraphColBERT (FiLM-фьюжн)',           acc_film),
]
df = pd.DataFrame(rows_tbl, columns=['Модель', 'Accuracy']).sort_values('Accuracy', ascending=False)
df['Accuracy'] = df['Accuracy'].map(lambda x: f'{x:.4f}')
print(f'Test: {len(test_ok)} вопросов\n')
print(df.to_string(index=False))

In [ ]:
metrics = {
    'dataset':          'AI2D',
    'task':             'MCQ',
    'split':            'test',
    'metric':           'accuracy',
    'n_test':           len(test_ok),
    'n_train_used':     len(train_ok),
    'random_baseline':  0.25,
    'graph_only':       round(float(acc_graph), 4),
    'plain_colbert':    round(float(acc_plain), 4),
    'graphcolbert_film': round(float(acc_film), 4),
    'best_val_acc':     round(float(best_acc), 4),
    'colbert_model':    COLBERT_NAME,
    'graph_checkpoint': str(CKPT_GRAPH.relative_to(ROOT)),
    'G_DIM': G_DIM, 'D_CB': D_CB,
    'config': {'MAX_TRAIN': MAX_TRAIN, 'MAX_VAL': MAX_VAL, 'MAX_TEST': MAX_TEST,
               'OCR_MIN_CONF': OCR_MIN_CONF, 'MAX_OCR_WORDS': MAX_OCR_WORDS,
               'EPOCHS': EPOCHS, 'LR': LR, 'FILM_HIDDEN': FILM_HIDDEN},
}
(OUTPUT_DIR / 'metrics_all.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2),
                                             encoding='utf-8')
print(json.dumps(metrics, ensure_ascii=False, indent=2))